# 03 - Data Cleaning & Preprocessing

## Objective

Transform the validated raw marketing data into a clean, analysis-ready dataset.

### Topics
- Data quality review
- Missing value treatment
- Duplicate removal
- Outlier treatment (Winsorization)
- Business rule validation
- Feature scaling
- Save processed dataset


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw" / "marketing_mix_data.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW, parse_dates=["Week"])
print(df.shape)
df.head()


## 1. Remove Duplicate Records

In [ ]:

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Removed {before-after} duplicate rows")


## 2. Missing Value Treatment

In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy="median")
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print(df.isna().sum().sum(), "missing values remaining")


## 3. Business Rule Validation

In [ ]:

rules = {
    "Sales > 0": (df["Sales"] > 0).all(),
    "Revenue > 0": (df["Revenue"] > 0).all(),
    "Orders > 0": (df["Orders"] > 0).all(),
    "Price > 0": (df["Price"] > 0).all(),
    "Discount between 0 and 100": df["Discount"].between(0,100).all()
}

pd.DataFrame({
    "Rule": rules.keys(),
    "Passed": rules.values()
})


## 4. Winsorize Outliers

In [ ]:

def winsorize_iqr(series):
    q1 = series.quantile(.25)
    q3 = series.quantile(.75)
    iqr = q3-q1
    lower = q1-1.5*iqr
    upper = q3+1.5*iqr
    return series.clip(lower, upper)

for col in numeric_cols:
    df[col] = winsorize_iqr(df[col])

print("Outliers capped using IQR.")


## 5. Feature Scaling

In [ ]:

media_cols = [
    "Google_Search","Google_Display","Meta","Instagram",
    "YouTube","TV","Radio","Influencer","Affiliate","Email"
]

scaler = StandardScaler()

scaled = scaler.fit_transform(df[media_cols])

scaled_df = pd.DataFrame(
    scaled,
    columns=[c+"_zscore" for c in media_cols]
)

df = pd.concat([df, scaled_df], axis=1)

df.head()


## 6. Distribution Check

In [ ]:

fig, ax = plt.subplots(figsize=(10,4))
ax.hist(df["Sales"], bins=25)
ax.set_title("Sales Distribution")
plt.show()


## 7. Save Clean Dataset

In [ ]:

output = PROCESSED_DIR / "marketing_mix_cleaned.csv"

df.to_csv(output, index=False)

print("Saved:", output)


# Industry Notes

Typical production cleaning pipelines also include:

- KNN/MICE imputation
- Invalid category detection
- Unit standardization
- Time-series gap filling
- Feature drift checks
- Schema versioning
- Automated quality reports

These will be introduced later where appropriate.

---
